# GMTI-Net Colab: Full train, test, save & reload

This notebook automates a reproducible Colab workflow: clone or copy the repo, install deps, run tests, run training, save checkpoints to Drive, then load a saved checkpoint and run validation/inference to verify the saved model.

Run the cells in order. Edit the variables in the first code cell to match your GitHub repo or Drive layout.


In [ ]:
# Parameters: set these before running the notebook
REPO_URL = "https://github.com/your-username/your-repo.git"  # replace with your repo URL if using GitHub
BRANCH = "main"  # or the branch you pushed to
USE_GDRIVE_COPY = (
    False  # If True, the notebook will copy a zip from Drive instead of git cloning
)
GDRIVE_ZIP_PATH = "/content/drive/MyDrive/GMTI-Net.zip"  # path to uploaded zip on Drive (if using Drive copy)
CLONE_DIR = "/content/gmti_net"  # local path to clone/copy the repo into
# Training options
MAX_ITERS = (
    200  # set to a small value for quick smoke runs; increase for longer training
)
RESUME_FROM = (
    None  # set to 'checkpoints/latest.pth' to resume from latest or a specific path
)
SAVE_TO_DRIVE_DIR = "/content/drive/MyDrive/GMTI-Net-Checkpoints"  # where to copy checkpoints/visualizations on Drive

print("Parameters:")
print("REPO_URL=", REPO_URL)
print("USE_GDRIVE_COPY=", USE_GDRIVE_COPY)
print("CLONE_DIR=", CLONE_DIR)
print("MAX_ITERS=", MAX_ITERS)

: 

In [ ]:
# 1) Mount Google Drive (will prompt for authorization)
from google.colab import drive

drive.mount("/content/drive")

# Prepare repo: clone from GitHub or unzip from Drive depending on the parameter
import os, shutil

if os.path.exists("/content/gmti_net"):
    print("Removing existing repo folder /content/gmti_net")
    shutil.rmtree("/content/gmti_net")

if USE_GDRIVE_COPY:
    # Copy and unzip the repo zip from Drive
    if not os.path.exists(GDRIVE_ZIP_PATH):
        raise FileNotFoundError(f"Expected zip at {GDRIVE_ZIP_PATH} on Drive")
    !cp "{GDRIVE_ZIP_PATH}" /content/GMTI-Net.zip
    !unzip -q /content/GMTI-Net.zip -d /content
    # assume repo content is in /content/GMTI-Net or similar; move to CLONE_DIR
    if os.path.exists("/content/GMTI-Net"):
        os.rename("/content/GMTI-Net", CLONE_DIR)
else:
    # Clone from GitHub (fast if repo is public)
    !git clone --branch {BRANCH} {REPO_URL} {CLONE_DIR} || true

%cd {CLONE_DIR}
!git rev-parse --abbrev-ref HEAD || true

In [ ]:
# 2) Install dependencies
# This installs requirements; if you need a specific torch wheel, uncomment the torch install line and adjust for CUDA compatibility
!pip install -r requirements.txt
# Example to install a specific torch wheel (adjust CUDA version):
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Print torch CUDA availability and versions for debugging
import torch, sys

print("python", sys.version)
print("torch", torch.__version__, "cuda_available=", torch.cuda.is_available())
if torch.cuda.is_available():
    print(
        "cuda_device_count=",
        torch.cuda.device_count(),
        "current_device=",
        torch.cuda.current_device(),
    )

In [ ]:
# 3) Run the test suite to ensure code integrity before training
# This will run pytest; if you have many tests this may take a while.
!pytest -q || true

In [ ]:
# 4) Run training (short example).
# The training script accepts --max_iters; adjust MAX_ITERS variable above.
TRAIN_CMD = f"python train.py --config config.yaml --max_iters {MAX_ITERS}"
if RESUME_FROM:
    TRAIN_CMD += f" --resume {RESUME_FROM}"
print("Running:", TRAIN_CMD)
get_ipython().system(TRAIN_CMD)

In [ ]:
# 5) After training, list saved checkpoints and logs
import os

print("Checkpoints:")
print(
    "\n".join(sorted([p for p in os.listdir("checkpoints")]))
    if os.path.exists("checkpoints")
    else "No checkpoints"
)
print("Logs:")
print(os.path.exists("logs"))
# Display a small sample of the latest checkpoint metadata
import torch

latest = "checkpoints/latest.pth"
if os.path.exists(latest):
    try:
        from utils.io import safe_torch_load

        ckpt = safe_torch_load(latest, map_location="cpu", weights_only=False)
        print("Loaded checkpoint iteration:", ckpt.get("iteration"))
        print("Keys in checkpoint:", list(ckpt.keys()))
    except Exception as e:
        print("Failed to read checkpoint metadata:", e)
else:
    print("No latest checkpoint found at", latest)

In [ ]:
# 6) Save artifacts to Drive (checkpoints, visualizations, results)
import shutil, os

os.makedirs(SAVE_TO_DRIVE_DIR, exist_ok=True)
for name in ["checkpoints", "visualizations", "results", "logs"]:
    if os.path.exists(name):
        dest = os.path.join(SAVE_TO_DRIVE_DIR, name)
        print("Copying", name, "->", dest)
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(name, dest)
print("Saved artifacts to Drive at", SAVE_TO_DRIVE_DIR)

## Load a saved checkpoint and run validation / inference to verify the model


In [ ]:
# 7) Validate the latest checkpoint using the provided script
CKPT = "checkpoints/latest.pth"
if os.path.exists(CKPT):
    print("Running validate.py on", CKPT)
    !python validate.py --config config.yaml --checkpoint {CKPT}
else:
    print("No checkpoint to validate:", CKPT)

In [ ]:
# 8) Run inference on a small set (first validation sequence) and save results
OUTDIR = "colab_results"
os.makedirs(OUTDIR, exist_ok=True)
# Run inference using latest checkpoint (if present)
if os.path.exists(CKPT):
    !python inference.py --config config.yaml --input_dir {os.path.abspath('val')} --output_dir {OUTDIR} --checkpoint {CKPT} || true
    print("Inference outputs saved to", OUTDIR)
else:
    print("No checkpoint found, skipping inference")

## Optional: Programmatically load the model and run a single forward pass (verification)


In [ ]:
# 9) Programmatically load saved model weights and run a small forward pass on a sample from the val set
import os, torch, glob
from PIL import Image
from torchvision import transforms
from utils.io import safe_torch_load
from models.gmti_net import GMTINet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device=", device)
ckpt_path = "checkpoints/latest.pth"
if os.path.exists(ckpt_path):
    ckpt = safe_torch_load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt.get("ema", ckpt.get("model", None))
    if state is None:
        print("No model state found in checkpoint")
    else:
        # Build model from config (use config.yaml values)
        import yaml

        with open("config.yaml", "r") as f:
            cfg = yaml.safe_load(f)
        model = GMTINet(
            swin_depth=cfg["model"]["encoder"]["swin_depth"],
            swin_heads=cfg["model"]["encoder"]["swin_heads"],
            swin_window_size=cfg["model"]["encoder"]["swin_window_size"],
            swin_mlp_ratio=cfg["model"]["encoder"]["swin_mlp_ratio"],
            flow_refinement_iters=cfg["model"]["flow"]["refinement_iters"],
            use_deformable=cfg["model"]["flow"]["use_deformable"],
            transformer_blocks=cfg["model"]["transformer"]["blocks"],
            transformer_heads=cfg["model"]["transformer"]["heads"],
            transformer_dim=cfg["model"]["transformer"]["embed_dim"],
            transformer_mlp_ratio=cfg["model"]["transformer"]["mlp_ratio"],
        ).to(device)
        model.load_state_dict(state)
        model.eval()
        # Find a validation pair to run through the model
        val_dirs = sorted(glob.glob("val/vid_*"))
        if len(val_dirs) == 0:
            print(
                "No val sequences found under val/; create or copy a small val set to run a forward pass"
            )
        else:
            # load two frames from the first val sequence
            frs = sorted(glob.glob(os.path.join(val_dirs[0], "*.png")))[:2]
            if len(frs) < 2:
                print("Need at least 2 frames to run inference")
            else:
                tf = transforms.Compose([transforms.ToTensor()])
                L = tf(Image.open(frs[0]).convert("RGB")).unsqueeze(0).to(device)
                R = tf(Image.open(frs[1]).convert("RGB")).unsqueeze(0).to(device)
                with torch.no_grad():
                    pred, aux = model(L, R)
                print("Forward pass complete. Pred shape:", pred.shape)
else:
    print("No checkpoint found at", ckpt_path)

## Notes and next steps

- If you want a fully automated run (train -> save -> validate -> push artifacts to Drive), you can set `MAX_ITERS` and `SAVE_TO_DRIVE_DIR` at the top and run sequentially.
- If you prefer the notebook to auto-detect CUDA and install the exact torch wheel, tell me the CUDA version you want and I can modify the install cell to pick the right wheel.
- If you want, I'll add a small, synthetic demo dataset to `train/` and `val/` so the notebook can be run immediately without uploading data.
